In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, 27aa4c31-5907-4722-bfda-46134d7b9ef3, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, 27aa4c31-5907-4722-bfda-46134d7b9ef3, 4, Finished, Available, Finished, False)

18 projects found


In [3]:
all_prime_contracts = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling prime contracts for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            f"https://api.procore.com/rest/v2.0/companies/{COMPANY_ID}/projects/{project_id}/prime_contracts",
            headers=headers,
            params={"page": page, "per_page": 100}
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code} for {project_name}, skipping")
            break

        data = response.json()
        rows = data.get("data", [])

        if not rows:
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_prime_contracts.extend(rows)

        total_count = data.get("meta", {}).get("total_count", 0)
        if page * 100 >= total_count:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total prime contracts: {len(all_prime_contracts)}")

StatementMeta(, 27aa4c31-5907-4722-bfda-46134d7b9ef3, 5, Finished, Available, Finished, False)

Pulling prime contracts for: 1100 Fulton Street
Pulling prime contracts for: 11 ESSEX ST
Pulling prime contracts for: 337A & 337B West Broadway Rehabilitaion Work
Pulling prime contracts for: 360 Lexington 8th & 20th Floor
Pulling prime contracts for: 549 Munroe Av
Pulling prime contracts for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling prime contracts for: Boys & Girls Club
Pulling prime contracts for: EMBANKMENT PHASE II
Pulling prime contracts for: Embankment Phase III
Pulling prime contracts for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling prime contracts for: Lillipvt 45 Renwick St
Pulling prime contracts for: PCNA 711 11TH AVE
Pulling prime contracts for: Sandbox Test Project
Pulling prime contracts for: SaunaLounge 45 South 3 Street, Brooklyn, NY
Pulling prime contracts for: Standard Project Template
Pulling prime contracts for: SYMRISE - 15th & 16th Flr
Pulling prime contracts for: TEST - ABM SUBORDINATE
  Error 403 for TEST - ABM SUBORDINATE, skipping
P

In [4]:
import pandas as pd
import re

clean_rows = []
for row in all_prime_contracts:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_prime_contracts_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_prime_contracts_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, 27aa4c31-5907-4722-bfda-46134d7b9ef3, 6, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
